# F-OAL — Controlled Pretrained ViT Backbone Alignment Experiment (C2)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL) (Pinned Commit: `73e4ac1b4c965030973a2b77f05c57c2f44f49f5`)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Target Branch:** `experiment/c2-backbone-alignment`
- **Research Question (C2):** Isolate the effect of Pretrained ViT Checkpoint (ImageNet-1K only vs. ImageNet-21K + 1K fine-tune) while holding all other factors fixed:
  - **B0 Baseline Reference:** `vit_base_patch16_224.augreg_in1k` (ImageNet-1K AugReg, C1 P1 setup)
  - **B1 Official Backbone:** `vit_base_patch16_224.augreg2_in21k_ft_in1k` (ImageNet-21K pre-trained, fine-tuned on ImageNet-1K)
- **Strict Controls:**
  - Projection Equality: Identical random projection matrix $P$ (seed 42, $D=1000$, `official_uniform`, identical SHA256)
  - Preprocessing: Bicubic 224x224, Normalize(0.5, 0.5) (C1 setup, fixed)
  - Dataset & Split: CIFAR-100, 10 tasks x 10 classes, seed 42
  - Optimization: Online Ridge RLS, batch size 10, gamma 1.0, float64
- **B0 Reproducibility Guard:** B0 is compared directly to C1 P1 reference (Avg Incr Acc 83.28%, Last Acc 74.84%) to detect any environmental discrepancy before assessing B1.

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Please switch Colab runtime to GPU (A100 / V100 / L4 / T4) before running."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")
print(f"Compute Cap.     : {device_props.major}.{device_props.minor}")

free_bytes = shutil.disk_usage(".").free
print(f"Free Disk Space  : {free_bytes / (1024**3):.2f} GB")


In [ ]:
# === CELL 2: Clone Group Repository & Checkout C2 Branch ===
from pathlib import Path

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/c2-backbone-alignment"

if REPO_DIR.exists():
    print(f"Removing existing directory {REPO_DIR} for clean clone...")
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "-b", BRANCH_NAME,
    "https://github.com/caoban123/F-OAL.git", str(REPO_DIR)
], check=True)

commit_hash = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
print(f"SUCCESS: Cloned {BRANCH_NAME} at commit {commit_hash}")

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
# === CELL 3: Install Project Dependencies ===
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=str(REPO_DIR), check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(REPO_DIR), check=True)

import timm
import datasets
import wandb
print(f"timm version     : {timm.__version__}")
print(f"datasets version : {datasets.__version__}")
print(f"wandb version    : {wandb.__version__}")

print("Running preflight unit tests...")
subprocess.run([sys.executable, "-m", "pytest", "-v", "tests/test_c2_backbone.py"], cwd=str(REPO_DIR), check=True)


In [ ]:
# === CELL 4: Prepare CIFAR-100 Dataset ===
from src.foal.datasets import CIFAR100OnlineDataset

DATA_DIR = REPO_DIR / "artifacts/data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Initializing & verifying CIFAR-100 dataset...")
ds_handler = CIFAR100OnlineDataset(root=str(DATA_DIR), seed=42)
train_src = ds_handler.get_train_source()
test_src = ds_handler.get_test_source()
print(f"Train samples count: {len(train_src)}")
print(f"Test samples count : {len(test_src)}")
assert len(train_src) == 50000, f"Expected 50000 train samples, got {len(train_src)}"
assert len(test_src) == 10000, f"Expected 10000 test samples, got {len(test_src)}"


In [ ]:
# === CELL 5: Configuration & Projection Equality Validation ===
import yaml
from src.foal.projection import SmoothProjection

cfg_b0_p = REPO_DIR / "configs/c2_b0_augreg_in1k.yaml"
cfg_b1_p = REPO_DIR / "configs/c2_b1_augreg2_in21k_ft_in1k.yaml"

with open(cfg_b0_p, "r", encoding="utf-8") as f:
    cfg_b0 = yaml.safe_load(f)
with open(cfg_b1_p, "r", encoding="utf-8") as f:
    cfg_b1 = yaml.safe_load(f)

print(f"B0 Backbone: {cfg_b0['model']['backbone']}")
print(f"B1 Backbone: {cfg_b1['model']['backbone']}")

# Verify Projection Equality
proj_b0 = SmoothProjection(768, 1000, seed=42, initialization='official_uniform')
proj_b1 = SmoothProjection(768, 1000, seed=42, initialization='official_uniform')
assert torch.equal(proj_b0.weight, proj_b1.weight), "FATAL: Projection weights differ between B0 and B1!"
p_sha = proj_b0.get_metadata()['matrix_sha256']
print(f"Projection Equality VERIFIED! Both use matrix P SHA256: {p_sha[:16]}...")


In [ ]:
# === CELL 6: Verify Both Pretrained ViT Checkpoints ===
from src.foal.backbone import FrozenViT

print("Loading & verifying B0 Checkpoint (augreg_in1k)...")
b0_vit = FrozenViT("vit_base_patch16_224.augreg_in1k", pretrained=True, feature_fusion=True)
b0_meta = b0_vit.get_backbone_metadata()
print(f"  B0 Tag: {b0_meta['pretrained_tag']}")
print(f"  B0 Hub ID: {b0_meta['hf_hub_id']}")
print(f"  B0 Feature Dim: {b0_meta['feature_dim']}")
print(f"  B0 Num Blocks: {b0_meta['num_blocks']}")
assert b0_meta["pretrained_tag"] == "augreg_in1k"
del b0_vit

print("\nLoading & verifying B1 Checkpoint (augreg2_in21k_ft_in1k)...")
b1_vit = FrozenViT("vit_base_patch16_224.augreg2_in21k_ft_in1k", pretrained=True, feature_fusion=True)
b1_meta = b1_vit.get_backbone_metadata()
print(f"  B1 Tag: {b1_meta['pretrained_tag']}")
print(f"  B1 Hub ID: {b1_meta['hf_hub_id']}")
print(f"  B1 Feature Dim: {b1_meta['feature_dim']}")
print(f"  B1 Num Blocks: {b1_meta['num_blocks']}")
assert b1_meta["pretrained_tag"] == "augreg2_in21k_ft_in1k"
del b1_vit
print("\nSUCCESS: Both pretrained ViT checkpoints verified on disk!")


In [ ]:
# === CELL 7: Matched-Input Backbone Diagnostics ===
DIAG_OUT_DIR = REPO_DIR / "results/c2_diagnostics"
DIAG_OUT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m", "scripts.run_c2_diagnostics",
    "--output-dir", str(DIAG_OUT_DIR),
    "--device", "cuda",
    "--num-samples", "200"
], cwd=str(REPO_DIR), check=True)


In [ ]:
# === CELL 8: Level A Pilot (2 Tasks) for B0 and B1 ===
PILOT_DIR = REPO_DIR / "results/c2_pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

print("--- Running B0 Pilot (2 Tasks) ---")
subprocess.run([
    sys.executable, "-m", "scripts.train",
    "--config", "configs/c2_b0_augreg_in1k.yaml",
    "--output-dir", str(PILOT_DIR / "pilot_b0"),
    "--max-tasks", "2",
    "--limit-train-per-task", "100",
    "--limit-test-per-task", "100",
    "--device", "cuda"
], cwd=str(REPO_DIR), check=True)

print("--- Running B1 Pilot (2 Tasks) ---")
subprocess.run([
    sys.executable, "-m", "scripts.train",
    "--config", "configs/c2_b1_augreg2_in21k_ft_in1k.yaml",
    "--output-dir", str(PILOT_DIR / "pilot_b1"),
    "--max-tasks", "2",
    "--limit-train-per-task", "100",
    "--limit-test-per-task", "100",
    "--device", "cuda"
], cwd=str(REPO_DIR), check=True)
print("SUCCESS: Level A Pilot completed for both B0 and B1!")


In [ ]:
# === CELL 9: Full 10-Task CIFAR-100 GPU Experiment ===
FULL_OUT_DIR = REPO_DIR / "results/c2_cifar100_full"
CACHE_DIR = REPO_DIR / "results/c2_feature_cache"
FULL_OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Weights & Biases configuration for CoreWeave Forge
WANDB_ENDPOINT = "https://api.forge.coreweave.com"
WANDB_ENTITY = "caoban170106-tr-ng-i-h-c-khoa-h-c-t-nhi-n-hqg-hcm"
WANDB_PROJECT = "hcmus-foal-reproduction"
os.environ["WANDB_BASE_URL"] = WANDB_ENDPOINT

use_wandb = False
try:
    from google.colab import userdata
    wb_key = userdata.get('WANDB_API_KEY')
    if wb_key:
        wandb.login(key=wb_key, host=WANDB_ENDPOINT)
        use_wandb = True
except Exception:
    pass
if not use_wandb and os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'], host=WANDB_ENDPOINT)
    use_wandb = True

print(f"W&B Tracking: {'ONLINE' if use_wandb else 'OFFLINE (saved locally for packaging)'}")
print(f"  Endpoint : {WANDB_ENDPOINT}")
print(f"  Entity   : {WANDB_ENTITY}")
print(f"  Project  : {WANDB_PROJECT}")

runs = [
    ("configs/c2_b0_augreg_in1k.yaml", "b0_augreg_in1k", "c2_cifar100_b0_augreg_in1k_seed42"),
    ("configs/c2_b1_augreg2_in21k_ft_in1k.yaml", "b1_augreg2_in21k_ft_in1k", "c2_cifar100_b1_augreg2_in21k_ft_in1k_seed42"),
]

for cfg_file, run_subdir, run_name in runs:
    out_path = FULL_OUT_DIR / run_subdir
    print(f"\n{'='*70}\nRUNNING: {run_name} -> {out_path}\n{'='*70}")
    cmd = [
        sys.executable, "-m", "scripts.train",
        "--config", cfg_file,
        "--output-dir", str(out_path),
        "--device", "cuda",
        "--run-name", run_name,
        "--feature-cache-dir", str(CACHE_DIR),
        "--wandb-project", WANDB_PROJECT,
        "--wandb-entity", WANDB_ENTITY,
        "--wandb-base-url", WANDB_ENDPOINT,
        "--wandb-group", "c2_backbone_alignment",
    ]
    if use_wandb:
        cmd.append("--wandb")
    else:
        cmd.extend(["--wandb", "--wandb-offline"])
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
print("\nSUCCESS: Both full 10-task experiments completed!")


In [ ]:
# === CELL 10: Results Recalculation, C1 P1 Alignment & Comparison Table ===
import json
import pandas as pd

b0_metrics = json.loads((FULL_OUT_DIR / "b0_augreg_in1k/metrics.json").read_text(encoding='utf-8'))
b1_metrics = json.loads((FULL_OUT_DIR / "b1_augreg2_in21k_ft_in1k/metrics.json").read_text(encoding='utf-8'))

# C1 P1 Verified Reference (Same backbone augreg_in1k, same projection official_uniform, seed 42)
c1_p1_ref_avg_acc = 0.8328
c1_p1_ref_last_acc = 0.7484

b0_avg_acc = b0_metrics['average_incremental_accuracy']
b0_last_acc = b0_metrics['last_accuracy']
b1_avg_acc = b1_metrics['average_incremental_accuracy']
b1_last_acc = b1_metrics['last_accuracy']

delta_b0_vs_c1 = (b0_avg_acc - c1_p1_ref_avg_acc) * 100
delta_b1_vs_b0 = (b1_avg_acc - b0_avg_acc) * 100

summary = [
    {
        "Condition": "C1 P1 Official Uniform (Verified Ref)",
        "Backbone Pretrain": "ImageNet-1K (AugReg)",
        "Avg Incr Acc (%)": f"{c1_p1_ref_avg_acc * 100:.2f}%",
        "Last Acc (%)": f"{c1_p1_ref_last_acc * 100:.2f}%",
        "Delta vs C1 P1": "Reference (0.00 pp)",
        "Status / Interpretation": "Historical C1 Full GPU Reference",
    },
    {
        "Condition": "C2 B0 Baseline Checkpoint (Current)",
        "Backbone Pretrain": "ImageNet-1K (AugReg)",
        "Avg Incr Acc (%)": f"{b0_avg_acc * 100:.2f}%",
        "Last Acc (%)": f"{b0_last_acc * 100:.2f}%",
        "Delta vs C1 P1": f"{delta_b0_vs_c1:+.2f} pp",
        "Status / Interpretation": "PASS: Consistent with C1 P1" if abs(delta_b0_vs_c1) <= 0.50 else "WARNING: Discrepancy >0.5pp with C1 P1! Investigate GPU/env non-determinism.",
    },
    {
        "Condition": "C2 B1 Official Checkpoint (Current)",
        "Backbone Pretrain": "ImageNet-21K + 1K ft",
        "Avg Incr Acc (%)": f"{b1_avg_acc * 100:.2f}%",
        "Last Acc (%)": f"{b1_last_acc * 100:.2f}%",
        "Delta vs C1 P1": f"{(b1_avg_acc - c1_p1_ref_avg_acc) * 100:+.2f} pp",
        "Status / Interpretation": f"Backbone effect: {delta_b1_vs_b0:+.2f} pp vs B0",
    },
]
df_res = pd.DataFrame(summary)
print("\n" + "=" * 90)
print("--- C2 EXPERIMENTAL RESULTS COMPARISON & REPRODUCIBILITY AUDIT ---")
print("=" * 90)
print(df_res.to_string(index=False))

print(f"\nReproducibility & Scientific Diagnostics:")
print(f"1. B0 vs C1 P1 Delta: {delta_b0_vs_c1:+.2f} pp (Threshold: +/-0.50 pp)")
if abs(delta_b0_vs_c1) > 0.50:
    print("   [ALERT] B0 diverges from C1 P1 reference! Check CUDA/cuDNN version or data loader ordering.")
else:
    print("   [OK] B0 successfully reproduces C1 P1 within acceptable GPU non-determinism limits.")

print(f"2. Controlled ViT Backbone Effect (B1 - B0):")
print(f"   Average Incremental Accuracy Delta: {delta_b1_vs_b0:+.2f} pp")
print(f"   Last Task Accuracy Delta           : {(b1_last_acc - b0_last_acc) * 100:+.2f} pp")


In [ ]:
# === CELL 11: Package Results into ZIP & Download ===
import zipfile

ZIP_PATH = Path("/content/foal_c2_backbone_results.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # Package results directory
    for p in REPO_DIR.glob("results/c2_*/**/*"):
        if p.is_file() and not p.name.endswith(".pyc"):
            zf.write(p, arcname=p.relative_to(REPO_DIR))
    # Package wandb logs if present
    for p in REPO_DIR.glob("wandb/offline-run-*/**/*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(REPO_DIR))

print(f"Packaged {ZIP_PATH} (Size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB)")

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("Triggered automatic browser download.")
except Exception as e:
    print(f"Note: Manual download available at {ZIP_PATH}")
